# Query observed redshifts from IPAC/NED

This notebook queries the NED redshift table for each galaxy and selects one observed redshift measurement per target.

Selection rule:
- prefer rows whose `Reference Frame` contains `Heliocentric`
- exclude rows without a published redshift
- break ties using smaller reported redshift uncertainty, then newer `Refcode`

The loop waits 1.1 seconds between requests to follow NED's automated-query guidance.

In [1]:
# 1) list of galaxies
targets = """
NGC4064 NGC4189 NGC4192 NGC4216 NGC4222 NGC4254
NGC4293 NGC4294 NGC4298 NGC4302 NGC4321 NGC4330
NGC4351 NGC4380 NGC4383 NGC4388 NGC4394 NGC4396
NGC4405 NGC4402 NGC4419 NGC4424 NGC4450 IC3392
NGC4457 NGC4501 NGC4522 NGC4535 NGC4548 NGC4567
NGC4568 NGC4569 NGC4579 NGC4580 NGC4606 NGC4607
NGC4654 NGC4689 NGC4694 NGC4698
""".split()

targets

['NGC4064',
 'NGC4189',
 'NGC4192',
 'NGC4216',
 'NGC4222',
 'NGC4254',
 'NGC4293',
 'NGC4294',
 'NGC4298',
 'NGC4302',
 'NGC4321',
 'NGC4330',
 'NGC4351',
 'NGC4380',
 'NGC4383',
 'NGC4388',
 'NGC4394',
 'NGC4396',
 'NGC4405',
 'NGC4402',
 'NGC4419',
 'NGC4424',
 'NGC4450',
 'IC3392',
 'NGC4457',
 'NGC4501',
 'NGC4522',
 'NGC4535',
 'NGC4548',
 'NGC4567',
 'NGC4568',
 'NGC4569',
 'NGC4579',
 'NGC4580',
 'NGC4606',
 'NGC4607',
 'NGC4654',
 'NGC4689',
 'NGC4694',
 'NGC4698']

In [2]:
# 2) helper functions for querying NED
import csv
import html
import io
import math
import random
import re
import time
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path
from IPython.display import HTML, display

NED_DATASEARCH_URL = "https://ned.ipac.caltech.edu/cgi-bin/datasearch"
NED_OBJSEARCH_URL = "https://ned.ipac.caltech.edu/cgi-bin/objsearch"
REQUEST_PAUSE_S = 1.1
REQUEST_TIMEOUT_S = 12
MAX_RETRIES = 3
INCLUDE_EBMV = True
OUTPUT_CSV = Path("ned_observed_redshifts_20260310.csv")


def normalize_target_name(name):
    text = re.sub(r"\s+", " ", str(name).strip())
    match = re.fullmatch(r"([A-Za-z]+)\s*0*([0-9]+[A-Za-z-]*)", text)
    if match:
        prefix, suffix = match.groups()
        return f"{prefix.upper()} {suffix}"
    return text.upper()


def fetch_url_bytes(url, timeout=REQUEST_TIMEOUT_S, retries=MAX_RETRIES):
    last_exc = None
    for attempt in range(retries + 1):
        try:
            request = urllib.request.Request(
                url,
                headers={
                    "User-Agent": "Mozilla/5.0 (compatible; NED-query-script/1.0)",
                    "Accept": "application/xml,text/xml,text/plain,*/*",
                    "Connection": "close",
                },
            )
            with urllib.request.urlopen(request, timeout=timeout) as response:
                return response.read()
        except Exception as exc:
            last_exc = exc
            if attempt < retries:
                base_wait = 1.2 * (2 ** attempt)
                jitter = random.uniform(0.2, 1.0)
                time.sleep(base_wait + jitter)
    raise RuntimeError(f"request failed after {retries + 1} attempts: {last_exc}")


def build_redshift_query_url(target):
    params = {
        "objname": normalize_target_name(target),
        "search_type": "Redshifts",
        "of": "xml_main",
    }
    return f"{NED_DATASEARCH_URL}?{urllib.parse.urlencode(params)}"


def build_basic_data_query_url(target):
    params = {
        "objname": normalize_target_name(target),
        "extend": "no",
        "of": "xml_all",
    }
    return f"{NED_OBJSEARCH_URL}?{urllib.parse.urlencode(params)}"


def fetch_redshift_rows(target, timeout=REQUEST_TIMEOUT_S):
    url = build_redshift_query_url(target)
    xml_bytes = fetch_url_bytes(url, timeout=timeout)

    root = ET.fromstring(xml_bytes)
    status = ""
    for info in root.findall(".//INFO"):
        if info.attrib.get("name") == "QUERY_STATUS":
            status = info.attrib.get("value", "")
            break

    if status and status.upper() != "OK":
        raise RuntimeError(f"NED returned QUERY_STATUS={status!r}")

    table = root.find(".//TABLE")
    if table is None:
        return [], url

    field_names = [field.attrib.get("name", "").strip() for field in table.findall("FIELD")]
    rows = []
    for tr in table.findall(".//TR"):
        values = []
        for td in tr.findall("TD"):
            values.append("".join(td.itertext()).strip())
        if len(values) < len(field_names):
            values.extend([""] * (len(field_names) - len(values)))
        row = dict(zip(field_names, values[: len(field_names)]))
        rows.append(row)

    return rows, url


def fetch_basic_data_row(target, timeout=REQUEST_TIMEOUT_S):
    url = build_basic_data_query_url(target)
    xml_bytes = fetch_url_bytes(url, timeout=timeout)
    root = ET.fromstring(xml_bytes)

    basic_tables = root.findall(".//TABLE[@ID='NED_BasicDataTable']")
    if not basic_tables:
        return {}, url

    fallback_row = {}
    for table in basic_tables:
        field_names = [field.attrib.get("name", "").strip() for field in table.findall("FIELD")]
        tr = table.find(".//TR")
        if tr is None:
            continue

        values = ["".join(td.itertext()).strip() for td in tr.findall("TD")]
        if len(values) < len(field_names):
            values.extend([""] * (len(field_names) - len(values)))
        row = dict(zip(field_names, values[: len(field_names)]))

        if not fallback_row:
            fallback_row = row

        ebmv_value = row.get("gal_extinc_E(B-V)", "")
        if str(ebmv_value).strip():
            return row, url

    return fallback_row, url


def extract_ebmv_from_basic_data(basic_row):
    if not basic_row:
        return ""
    if "gal_extinc_E(B-V)" in basic_row:
        return basic_row.get("gal_extinc_E(B-V)", "")

    for key, value in basic_row.items():
        compact = key.lower().replace(" ", "")
        if "e(b-v)" in compact or "ebmv" in compact:
            return value

    return ""


def parse_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return math.nan


def refcode_year(refcode):
    match = re.match(r"(\d{4})", refcode or "")
    return int(match.group(1)) if match else -1


def choose_observed_redshift(rows):
    candidates = [row for row in rows if row.get("Published Redshift", "").strip()]
    if not candidates:
        return None, []

    def reference_rank(row):
        reference_frame = row.get("Reference Frame", "").strip().lower()
        if "helio" in reference_frame:
            return 0
        if not reference_frame:
            return 1
        return 2

    def qualifier_rank(row):
        notes = " ".join([
            row.get("Qualifiers", ""),
            row.get("Comments", ""),
        ]).lower()
        return 1 if "uncertain origin" in notes else 0

    def uncertainty_rank(row):
        uncertainty = parse_float(row.get("Published Redshift Uncertainty", ""))
        if math.isnan(uncertainty) or uncertainty <= 0:
            return math.inf
        return uncertainty

    ranked = sorted(
        candidates,
        key=lambda row: (
            reference_rank(row),
            qualifier_rank(row),
            uncertainty_rank(row),
            -refcode_year(row.get("Refcode", "")),
        ),
    )
    return ranked[0], ranked


def display_results_table(rows):
    if not rows:
        display(HTML("<p>No results.</p>"))
        return

    columns = [
        "target",
        "selected_redshift",
        "selected_velocity_km_s",
        "ebmv",
        "ebmv_status",
        "reference_frame",
        "redshift_uncertainty",
        "refcode",
        "heliocentric_matches",
        "total_measurements",
        "status",
    ]

    head = "".join(f"<th>{html.escape(column)}</th>" for column in columns)
    body = []
    for row in rows:
        cells = "".join(
            f"<td>{html.escape(str(row.get(column, '')))}</td>" for column in columns
        )
        body.append(f"<tr>{cells}</tr>")

    table_html = (
        "<table>"
        "<thead><tr>" + head + "</tr></thead>"
        "<tbody>" + "".join(body) + "</tbody>"
        "</table>"
    )
    display(HTML(table_html))

In [3]:
# 3) run redshift + E(B-V) queries for all galaxies in parallel
from concurrent.futures import ThreadPoolExecutor, as_completed

results = []

MAX_TARGETS = None  # set to an integer (e.g., 10) for a quick test run
MAX_WORKERS = 4     # keep moderate to avoid overloading NED
FAILED_RETRY_PASSES = 1
run_targets = targets if MAX_TARGETS is None else targets[:MAX_TARGETS]

print(
    f"Starting {len(run_targets)} targets | include_ebmv={INCLUDE_EBMV} | "
    f"timeout={REQUEST_TIMEOUT_S}s retries={MAX_RETRIES} | workers={MAX_WORKERS}",
    flush=True,
)


def process_target(index, target):
    normalized_target = normalize_target_name(target)
    try:
        rows, query_url = fetch_redshift_rows(target)
        best_row, ranked_rows = choose_observed_redshift(rows)
        heliocentric_matches = sum(
            1
            for row in rows
            if "helio" in row.get("Reference Frame", "").strip().lower()
        )

        ebmv = ""
        basic_query_url = build_basic_data_query_url(target)
        if INCLUDE_EBMV:
            try:
                basic_row, basic_query_url = fetch_basic_data_row(target)
                ebmv = extract_ebmv_from_basic_data(basic_row)
                ebmv_status = "ok" if ebmv else "missing"
            except Exception as exc:
                ebmv_status = f"error: {exc}"
        else:
            ebmv_status = "skipped"

        result_row = {
            "__index": index,
            "target": normalized_target,
            "query_url": query_url,
            "basic_query_url": basic_query_url,
            "total_measurements": len(rows),
            "heliocentric_matches": heliocentric_matches,
            "status": "ok" if best_row else "no published redshift found",
            "ebmv_status": ebmv_status,
            "selected_redshift": "",
            "redshift_uncertainty": "",
            "selected_velocity_km_s": "",
            "reference_frame": "",
            "qualifiers": "",
            "comments": "",
            "refcode": "",
            "ebmv": ebmv,
        }

        if best_row:
            result_row.update(
                {
                    "selected_redshift": best_row.get("Published Redshift", ""),
                    "redshift_uncertainty": best_row.get("Published Redshift Uncertainty", ""),
                    "selected_velocity_km_s": best_row.get("Published Velocity", ""),
                    "reference_frame": best_row.get("Reference Frame", ""),
                    "qualifiers": best_row.get("Qualifiers", ""),
                    "comments": best_row.get("Comments", ""),
                    "refcode": best_row.get("Refcode", ""),
                }
            )

        return result_row
    except Exception as exc:
        return {
            "__index": index,
            "target": normalized_target,
            "query_url": build_redshift_query_url(target),
            "basic_query_url": build_basic_data_query_url(target),
            "total_measurements": 0,
            "heliocentric_matches": 0,
            "status": f"error: {exc}",
            "ebmv_status": "not attempted",
            "selected_redshift": "",
            "redshift_uncertainty": "",
            "selected_velocity_km_s": "",
            "reference_frame": "",
            "qualifiers": "",
            "comments": "",
            "refcode": "",
            "ebmv": "",
        }


pending = [(index, target) for index, target in enumerate(run_targets, start=1)]
all_rows = {}

for pass_index in range(FAILED_RETRY_PASSES + 1):
    if not pending:
        break

    if pass_index > 0:
        print(f"Retry pass {pass_index}: {len(pending)} targets", flush=True)
        time.sleep(3.0)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_map = {
            executor.submit(process_target, index, target): (index, target)
            for index, target in pending
        }

        pending_next = []
        for future in as_completed(future_map):
            index, target = future_map[future]
            try:
                row = future.result()
                print(
                    f"[{index}/{len(run_targets)}] {target} - done | "
                    f"status={row['status']} | ebmv_status={row['ebmv_status']}",
                    flush=True,
                )
            except Exception as exc:
                row = {
                    "__index": index,
                    "target": normalize_target_name(target),
                    "query_url": build_redshift_query_url(target),
                    "basic_query_url": build_basic_data_query_url(target),
                    "total_measurements": 0,
                    "heliocentric_matches": 0,
                    "status": f"error: {exc}",
                    "ebmv_status": "not attempted",
                    "selected_redshift": "",
                    "redshift_uncertainty": "",
                    "selected_velocity_km_s": "",
                    "reference_frame": "",
                    "qualifiers": "",
                    "comments": "",
                    "refcode": "",
                    "ebmv": "",
                }

            all_rows[index] = row
            needs_retry = (
                pass_index < FAILED_RETRY_PASSES
                and (
                    str(row.get("status", "")).lower().startswith("error:")
                    or str(row.get("ebmv_status", "")).lower().startswith("error:")
                )
            )
            if needs_retry:
                pending_next.append((index, target))

        pending = pending_next

results = [all_rows[index] for index in sorted(all_rows)]
for row in results:
    row.pop("__index", None)

print(f"Finished {len(results)} targets")
display_results_table(results)
results[:5]

Starting 40 targets | include_ebmv=True | timeout=12s retries=3 | workers=4
[1/40] NGC4064 - done | status=ok | ebmv_status=ok
[2/40] NGC4189 - done | status=ok | ebmv_status=ok
[4/40] NGC4216 - done | status=ok | ebmv_status=ok
[3/40] NGC4192 - done | status=ok | ebmv_status=ok
[5/40] NGC4222 - done | status=ok | ebmv_status=ok
[7/40] NGC4293 - done | status=ok | ebmv_status=ok
[6/40] NGC4254 - done | status=ok | ebmv_status=ok
[8/40] NGC4294 - done | status=ok | ebmv_status=ok
[10/40] NGC4302 - done | status=ok | ebmv_status=ok
[9/40] NGC4298 - done | status=ok | ebmv_status=ok
[11/40] NGC4321 - done | status=ok | ebmv_status=ok
[12/40] NGC4330 - done | status=ok | ebmv_status=ok
[13/40] NGC4351 - done | status=ok | ebmv_status=ok
[14/40] NGC4380 - done | status=ok | ebmv_status=ok
[15/40] NGC4383 - done | status=ok | ebmv_status=ok
[16/40] NGC4388 - done | status=ok | ebmv_status=ok
[17/40] NGC4394 - done | status=ok | ebmv_status=ok
[18/40] NGC4396 - done | status=ok | ebmv_status=

target,selected_redshift,selected_velocity_km_s,ebmv,ebmv_status,reference_frame,redshift_uncertainty,refcode,heliocentric_matches,total_measurements,status
NGC 4064,0.003025,907,0.019,ok,Heliocentric velocity or redshift,0.000009,2005ApJS..160..149S,24,25,ok
NGC 4189,0.006997,2098,0.029,ok,Heliocentric velocity or redshift,0.000011,2005SDSS4.C...0000:,42,43,ok
NGC 4192,-0.000474,-141,0.031,ok,Heliocentric velocity or redshift,0.000013,1991RC3.9.C...0000d,40,41,ok
NGC 4216,0.000437,131,0.028,ok,Heliocentric velocity or redshift,0.000013,1991RC3.9.C...0000d,37,38,ok
NGC 4222,0.000767,230,0.028,ok,Heliocentric velocity or redshift,0.000002,2005ApJS..160..149S,36,37,ok
NGC 4254,0.008026,2406,0.034,ok,Heliocentric velocity or redshift,0.000002,2005ApJS..160..149S,58,59,ok
NGC 4293,0.003072,921,0.035,ok,Heliocentric velocity or redshift,0.000021,2005ApJS..160..149S,29,30,ok
NGC 4294,0.001077,323,0.030,ok,Heliocentric velocity or redshift,0.000005,2005ApJS..160..149S,44,45,ok
NGC 4298,0.003740,1121,0.031,ok,Heliocentric velocity or redshift,0.000014,2006SDSS5.C...0000:,42,43,ok
NGC 4302,0.003696,1108,0.031,ok,Heliocentric velocity or redshift,0.000017,1991RC3.9.C...0000d,40,41,ok


[{'target': 'NGC 4064',
  'query_url': 'https://ned.ipac.caltech.edu/cgi-bin/datasearch?objname=NGC+4064&search_type=Redshifts&of=xml_main',
  'basic_query_url': 'https://ned.ipac.caltech.edu/cgi-bin/objsearch?objname=NGC+4064&extend=no&of=xml_all',
  'total_measurements': 25,
  'heliocentric_matches': 24,
  'status': 'ok',
  'ebmv_status': 'ok',
  'selected_redshift': '0.003025',
  'redshift_uncertainty': '0.000009',
  'selected_velocity_km_s': '907',
  'reference_frame': 'Heliocentric velocity or redshift',
  'qualifiers': 'From reprocessed raw data',
  'comments': "'Quality' Good.  Arecibo data.",
  'refcode': '2005ApJS..160..149S',
  'ebmv': '0.019'},
 {'target': 'NGC 4189',
  'query_url': 'https://ned.ipac.caltech.edu/cgi-bin/datasearch?objname=NGC+4189&search_type=Redshifts&of=xml_main',
  'basic_query_url': 'https://ned.ipac.caltech.edu/cgi-bin/objsearch?objname=NGC+4189&extend=no&of=xml_all',
  'total_measurements': 43,
  'heliocentric_matches': 42,
  'status': 'ok',
  'ebmv_st

In [4]:
# 4) summarize final results for all galaxies
if not results:
    print("No results yet. Run Cell 3 first.")
else:
    total = len(results)
    redshift_ok = sum(1 for r in results if str(r.get("status", "")).strip().lower() == "ok")
    redshift_error = sum(1 for r in results if str(r.get("status", "")).strip().lower().startswith("error:"))
    ebmv_ok = sum(1 for r in results if str(r.get("ebmv_status", "")).strip().lower() == "ok")
    ebmv_missing = sum(1 for r in results if str(r.get("ebmv_status", "")).strip().lower() == "missing")
    ebmv_error = sum(1 for r in results if str(r.get("ebmv_status", "")).strip().lower().startswith("error:"))

    print(f"Total galaxies processed: {total}")
    print(f"Redshift success (status=ok): {redshift_ok}")
    print(f"Redshift errors: {redshift_error}")
    print(f"E(B-V) success: {ebmv_ok}")
    print(f"E(B-V) missing: {ebmv_missing}")
    print(f"E(B-V) errors: {ebmv_error}")

    failed_targets = [r.get("target", "") for r in results if str(r.get("status", "")).lower().startswith("error:")]
    if failed_targets:
        print("\nTargets with redshift query errors:")
        print(", ".join(failed_targets))

    ebmv_failed_targets = [
        r.get("target", "")
        for r in results
        if str(r.get("ebmv_status", "")).lower().startswith("error:")
    ]
    if ebmv_failed_targets:
        print("\nTargets with E(B-V) query errors:")
        print(", ".join(ebmv_failed_targets))

Total galaxies processed: 40
Redshift success (status=ok): 40
Redshift errors: 0
E(B-V) success: 40
E(B-V) missing: 0
E(B-V) errors: 0
